# Đánh giá mô hình sinh caption trên Flickr30k

Đánh giá bốn mô hình trên cùng 1.000 ảnh thuộc test split với Greedy, Beam, Contrastive và Diverse Beam Search. Bảng so sánh dùng BLEU-1, BLEU-4, METEOR, ROUGE-L, CIDEr và SPICE. Diverse Beam được chấm riêng cho từng nhóm trên đủ 1.000 ảnh. Bảng so sánh lấy trung bình ba điểm theo từng chỉ số. Bảng caption giữ kết quả của cả ba nhóm.

Chạy ô cấu hình và ô xem caption để đọc kết quả của lượt hiện tại. Khi chủ động chạy lại, đặt một thư mục kết quả mới để giữ số liệu cũ. Trên Colab, trỏ cấu hình tới thư mục chứa mã nguồn, features và bốn checkpoint hiện hành.

In [ ]:
from pathlib import Path

# Run from the repository root or the notebooks directory.
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
assert (repo_root / 'src/eval/evaluate.py').is_file(), repo_root

checkpoint_root = repo_root / 'outputs/checkpoints'
feature_root = repo_root / 'outputs/features'
csv_path = repo_root / 'data/results.csv'
split_path = repo_root / 'configs/splits.json'
output_dir = repo_root / 'outputs/evaluation'
active_run_dir = repo_root / 'outputs/evaluation_latest_20261126'
# Select a working JDK if the Java on PATH is unavailable.
java_home = Path(r"C:\Program Files\JetBrains\IntelliJ IDEA 2025.3\jbr")

## Chạy đánh giá

Ô này chạy suy luận từ bốn file best.pt trong thư mục checkpoint đã cấu hình. Dùng kernel của môi trường đã cài dependencies của dự án và Java cho METEOR, SPICE.

In [ ]:
import os
import json
import subprocess
import sys

test_names = json.loads(split_path.read_text(encoding='utf-8'))['test']
assert len(test_names) == len(set(test_names)) == 1000
assert not list(output_dir.glob('*_captions.csv')), 'Set output_dir to a new directory before rerunning.'

env = os.environ.copy()
if (java_home / 'bin/java.exe').is_file():
    env['PATH'] = str(java_home / 'bin') + os.pathsep + env['PATH']

subprocess.run([
    sys.executable, '-m', 'src.eval.evaluate',
    '--csv_path', str(csv_path),
    '--splits', str(split_path),
    '--features_root', str(feature_root),
    '--checkpoint_root', str(checkpoint_root),
    '--output_dir', str(output_dir),
    '--sample_size', '1000',
    '--seed', '42',
    '--device', 'cuda',
], cwd=repo_root, env=env, check=True)

## Bảng so sánh sáu chỉ số

Mỗi phương pháp được chấm trên cùng 1.000 ảnh, đối chiếu caption sinh ra với các caption tham chiếu Flickr30k. Điểm càng cao càng tốt theo tiêu chí của từng chỉ số. Thang điểm của CIDEr khác với BLEU và các chỉ số còn lại.

| Chỉ số | Tiêu chí |
| --- | --- |
| BLEU-1 | Mức trùng từ với caption tham chiếu, có phạt câu quá ngắn |
| BLEU-4 | Mức trùng các cụm từ dài từ một đến bốn từ |
| METEOR | Mức khớp từ có xét biến thể từ, từ đồng nghĩa và thứ tự |
| ROUGE-L | Độ dài dãy con chung dài nhất, giữ thứ tự từ |
| CIDEr | Mức đồng thuận với các tham chiếu qua n-gram có trọng số TF-IDF |
| SPICE | Mức khớp đối tượng, thuộc tính và quan hệ trong đồ thị ngữ nghĩa |

In [ ]:
import pandas as pd
from IPython.display import display

report_path = output_dir / 'comparison_6_scores.csv'
if report_path.exists():
    report = pd.read_csv(report_path)
    assert report['test_images'].eq(1000).all()
    print(f'{len(report)}/16 model × search combinations')
    display(report)
else:
    print('Bảng điểm đang chờ lượt đánh giá ghi kết quả.')

## Xem caption theo ảnh

Điền tên file vào `image_name` để so sánh các caption của cùng một ảnh. Để trống để xem ảnh đầu tiên có kết quả.

In [ ]:
import pandas as pd
from IPython.display import display

image_name = ''
caption_files = {file.name: file for file in active_run_dir.glob('*_captions.csv')}
caption_files.update({file.name: file for file in output_dir.glob('*_captions.csv')})
files = [caption_files[name] for name in sorted(caption_files)]
if files:
    samples = pd.concat([
        pd.read_csv(file, keep_default_na=False).assign(
            model=file.stem.removesuffix('_captions')
        )
        for file in files
    ], ignore_index=True)
    if not samples.empty:
        selected_image = image_name or samples.iloc[0]['image']
        selected = samples[samples['image'].eq(selected_image)]
        if selected.empty:
            print(f'Chưa có caption cho {selected_image}')
        else:
            print(selected_image)
            for reference in selected.iloc[0]['references'].splitlines():
                print('-', reference)
            display(selected.drop(columns=['image', 'references']).set_index('model'))
else:
    print('Chưa có file caption trong', output_dir)